In [ ]:
import pypsa
import pandas as pd
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
from _helpers import mock_snakemake
snakemake = mock_snakemake(
            "collect_figures",
            scenario="default" # default, penalty-sa, penalty-ea, penalty-nwa, penalty-oc
    )

In [ ]:
scenario = snakemake.wildcards.scenario

### Steel supply curve all regions (from model)

In [ ]:
process = "hbi" # or "eaf" # DISCLAIMER: Steel only supply curve "steel" from model not supported yet
scenario = "default" # default, penalty-sa, penalty-ea, penalty-nwa, penalty-oc
xlim = None

In [ ]:
if process == "hbi":
    trade_model = f"../../results/cost_year~2030/interone~hbi/intertwo~eaf-grid/final~steel/scenario~{scenario}/network.nc"
else:
    print("WARNING: Steel only supply curve 'steel' from model not supported yet")

### Read trade model


In [ ]:
n = pypsa.Network(trade_model)

In [ ]:
# Iron ore cost per ton of steel (w/o transport)
iron_ore_to_steel = snakemake.config["iron_ore"]["ore_to_steel_ratio"]
iron_ore_cost_notransport = n.generators[n.generators.carrier == "iron_ore"].marginal_cost.mean() * iron_ore_to_steel
print(f"Iron ore cost (w/o transport): {iron_ore_cost_notransport:.2f} €/t_steel")

In [ ]:
# Select process
interone = n.links[n.links.carrier == 'hbi']

In [ ]:
interone.loc[:,'region'] = interone['bus1'].str.replace('_hbi$', '', regex=True)

In [ ]:
# Marginal costs refer to bus0 (which is iron ore). To get the cost of steel (bus1), the process efficiency needs to be taken into account
iron_ore_to_steel = 1/interone.efficiency.mean()

interone.loc[:,'cost in €/t_steel'] = interone.marginal_cost.values * iron_ore_to_steel
interone.loc[:,'cost in €/t_steel incl. iron ore'] = interone['cost in €/t_steel'] + iron_ore_cost_notransport

In [ ]:
# Add quantity
interone.loc[:,'quantity in Mt_steel'] = interone.p_nom_max.values / iron_ore_to_steel / 1e6

In [ ]:
# Sort by cost for supply curve
interone_sorted = interone.sort_values('cost in €/t_steel incl. iron ore').reset_index(drop=True)

# Assign a color to each region
region_list = interone_sorted['region'].unique()
region_colors = snakemake.config["colors"]

cum_quantity_line = interone_sorted['quantity in Mt_steel'].cumsum()
cum_cost_line = interone_sorted['cost in €/t_steel incl. iron ore']

# Prepare for stacked area plot: for each link, plot a bar at its cost, colored by region
cum_quantity = 0
bar_lefts = []
bar_widths = []
bar_costs = []
bar_colors = []
for _, row in interone_sorted.iterrows():
    bar_lefts.append(cum_quantity)
    bar_widths.append(row['quantity in Mt_steel'])
    bar_costs.append(row['cost in €/t_steel incl. iron ore'])
    bar_colors.append(region_colors[row['region']])
    cum_quantity += row['quantity in Mt_steel']

In [ ]:
def plot_supply_curve(bar_lefts, bar_widths, bar_costs, bar_colors, region_colors, region_list, xlim=2500):

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(bar_lefts, bar_costs, width=bar_widths, color=bar_colors, align='edge', edgecolor='none', alpha=0.8)
    # Add the classic supply curve line
    
    ax.step(cum_quantity_line, cum_cost_line, where='pre', color='black', linewidth=1.2, label='Supply curve')

    # Add vertical line of current demand and REMIND demand
    current_demand = 2000  # Mt steel
    ax.axvline(current_demand, color='red', linestyle='--', label='Current demand (2020)')
    ax.annotate('Current demand (2025)', xy=(current_demand, 50), xytext=(current_demand+50, 150), rotation=90, color='red')
    remind_demand = 725  # Mt steel, REMIND global steel long-term hydrogen: 700-750 Mt
    ax.axvline(remind_demand, color='orange', linestyle='--', label='REMIND 2050 demand')
    ax.annotate('REMIND 2050 demand', xy=(remind_demand, 50), xytext=(remind_demand+50, 150), rotation=90, color='orange')

    ax.set_xlabel('Cumulative quantity (Mt)')
    ax.set_ylabel('Supply cost (€/t_steel)')
    ax.set_title(f'Combined {process} supply curve')
    ax.set_ylim(0, 1000)
    ax.set_xlim(0, sum(bar_widths))
    ax.set_xlim(0, xlim)
    ax.grid(axis='y', alpha=0.4, zorder=0)
    # Legend for regions
    handles = [plt.Rectangle((0,0),1,1, color=region_colors[reg]) for reg in region_list]
    handles.append(plt.Line2D([0], [0], color='black', linewidth=1.2, label='Supply curve'))
    labels = list(region_list) + ['Supply curve']
    ax.legend(handles, labels, title='Region', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(snakemake.input.global_supply_curve)
    plt.savefig(snakemake.input.global_supply_curve_png, dpi=300)
    plt.show()

    return

In [ ]:
plot_supply_curve(bar_lefts=bar_lefts, bar_widths=bar_widths, bar_costs=bar_costs, bar_colors=bar_colors, region_colors=region_colors, region_list=region_list, xlim=xlim)

### Steel supply curve (from single curves)

In [ ]:
process = "hbi" # eaf, hbi, eaf-grid, steel
include_ironorecost = True
ironorecost = 97*1.59
demand = True # Subtract local demand from supply curve
sort_over_all = True # "True" to sort over all regions, "False" to sort within each region
xlim = None
regions = snakemake.config["regions"].keys()

In [ ]:
df_all = pd.DataFrame()

for region in regions:
    
    if demand:
        fn = f"../../resources/supply_curves/cost_year~2030/{region}_{process}.csv"
    else:
        fn = f"../../resources/supply_curves_nodemand/cost_year~2030/{region}_{process}.csv"
    df = pd.read_csv(fn, index_col=0)
    df["region"] = region

    diff = df["demand [t]"].diff()
    diff[0] = diff[1]  # set first value to second to avoid NaN
    df["demand [t]"] = diff


    df_all = pd.concat([df_all, df], ignore_index=True)

if sort_over_all:
    df_all = df_all.sort_values('LCOX [EUR/t]').reset_index(drop=True)
else:
    pass

if include_ironorecost:
    df_all['LCOX [EUR/t]'] += ironorecost

In [ ]:
df_all[df_all["region"]=="Europe"]

In [ ]:
cum_quantity_line = df_all['demand [t]'].cumsum()/1e6  # Mt steel
cum_cost_line = df_all['LCOX [EUR/t]']

# Prepare for stacked area plot based on df_all
cum_quantity = 0
bar_lefts = []
bar_widths = []
bar_costs = []
bar_colors = []
for _, row in df_all.iterrows():
    bar_lefts.append(cum_quantity)
    bar_widths.append(row['demand [t]']/1e6)  # Mt steel
    bar_costs.append(row['LCOX [EUR/t]'])
    bar_colors.append(region_colors[row['region']])
    cum_quantity += row['demand [t]']/1e6  # Mt steel

In [ ]:
plot_supply_curve(bar_lefts=bar_lefts, bar_widths=bar_widths, bar_costs=bar_costs, bar_colors=bar_colors, region_colors=region_colors, region_list=region_list, xlim=xlim)